# 03 - Actualizar footprints en indice de vuelos

Tercera etapa del flujo Geosupport. Exporta desde el mosaic dataset las geometrías de las imagenes cargadas, las agrega al feature class de indice de vuelos PAO y recalcula el campo `Nombre_de_Vuelo` con el orden operativo usado en el notebook 009.

Ejecutar primero con `DRY_RUN = True`. Para escribir en la GDB y en el mosaico, cambiar a `DRY_RUN = False`.

In [3]:
from datetime import datetime
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'core').exists():
    for candidate in [Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / 'core').exists():
            PROJECT_ROOT = candidate
            break

if not (PROJECT_ROOT / 'core').exists():
    raise FileNotFoundError('No se encontro el folder core. Ejecuta el notebook desde la raiz del proyecto o desde flujo_geosupport_etapas.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FLOW_DIR = PROJECT_ROOT / 'flujo_geosupport_etapas'

import arcpy
arcpy.env.overwriteOutput = True

print('Proyecto raiz:', PROJECT_ROOT)
print('Folder flujo:', FLOW_DIR)

Proyecto raiz: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport
Folder flujo: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\flujo_geosupport_etapas


## Parametros

`02_load_results.csv` viene de la etapa 2. Se toman los `Name` cargados correctamente o ya existentes en el mosaico.

In [4]:
LOAD_RESULTS_CSV = FLOW_DIR / 'outputs' / 'etapa_02_carga_datastore_mosaico' / '02_load_results.csv'

PATH_MOSAIC_DATASET = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\CL MLP PAO Aereo Image Server_v2\SQLServer-amssclgis06_ArcGIS-Aereo.sde\OWD.CL_MLP_PAO_IF_Ortho_Geosupport"
FC_FOOTPRINT_INDICE = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO"

TEMP_GDB = PROJECT_ROOT / 'Dataset' / 'data.gdb'
if not TEMP_GDB.exists() and (PROJECT_ROOT / 'Dataset' / 'Datos.gdb').exists():
    TEMP_GDB = PROJECT_ROOT / 'Dataset' / 'Datos.gdb'
FC_FOOTPRINT_TEMP_NAME = 'footprints_nuevas_temp'

DRY_RUN = False
SKIP_EXISTING_TARGET_NAMES = True
RECALCULATE_ALL_NOMBRE_VUELO = True
UPDATE_MOSAIC_NOMBRE_VUELO = True

# run_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = FLOW_DIR / 'outputs' / 'etapa_03_actualizar_footprints_indice'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('CSV carga etapa 2:', LOAD_RESULTS_CSV)
print('Mosaic dataset:', PATH_MOSAIC_DATASET)
print('Feature indice vuelos:', FC_FOOTPRINT_INDICE)
print('GDB temporal:', TEMP_GDB)
print('DRY_RUN:', DRY_RUN)

CSV carga etapa 2: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\flujo_geosupport_etapas\outputs\etapa_02_carga_datastore_mosaico\02_load_results.csv
Mosaic dataset: \\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\CL MLP PAO Aereo Image Server_v2\SQLServer-amssclgis06_ArcGIS-Aereo.sde\OWD.CL_MLP_PAO_IF_Ortho_Geosupport
Feature indice vuelos: \\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO
GDB temporal: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\Dataset\Datos.gdb
DRY_RUN: False


## 1. Leer imagenes cargadas en etapa 2

In [5]:
if not LOAD_RESULTS_CSV.exists():
    raise FileNotFoundError(f'No existe el resultado de etapa 2: {LOAD_RESULTS_CSV}')

load_results_df = pd.read_csv(LOAD_RESULTS_CSV)
if 'Name' not in load_results_df.columns:
    raise ValueError('El CSV de etapa 2 debe contener la columna Name')

status_mask = pd.Series(True, index=load_results_df.index)
if 'overall_status' in load_results_df.columns:
    status_mask = load_results_df['overall_status'].astype(str).str.lower().isin(['ok', 'dry_run'])
if 'mosaic_add_status' in load_results_df.columns:
    status_mask = status_mask | load_results_df['mosaic_add_status'].astype(str).str.lower().isin(['added', 'already_exists', 'dry_run'])

loaded_df = load_results_df[status_mask & load_results_df['Name'].notna()].copy()
names_from_stage_2 = sorted(loaded_df['Name'].astype(str).unique().tolist())

print(f'Registros etapa 2: {len(load_results_df)}')
print(f'Nombres candidatos para exportar footprint: {len(names_from_stage_2)}')
display(loaded_df[['Name', 'destination_path', 'copy_status', 'mosaic_add_status', 'footprint_status', 'attribute_status', 'overall_status']].head(30))

Registros etapa 2: 33
Nombres candidatos para exportar footprint: 33


,Name,destination_path,copy_status,mosaic_add_status,footprint_status,attribute_status,overall_status
0,CL_MLP_PAO_IF_Ortho_25_12_28_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,already_exists,dry_run,dry_run,dry_run,dry_run
1,CL_MLP_PAO_IF_Ortho_25_12_10_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,already_exists,dry_run,dry_run,dry_run,dry_run
2,CL_MLP_PAO_IF_Ortho_25_12_20_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,already_exists,dry_run,dry_run,dry_run,dry_run
3,CL_MLP_PAO_IF_Ortho_26_01_03_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,already_exists,dry_run,dry_run,dry_run,dry_run
4,CL_MLP_PAO_IF_Ortho_26_01_17_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,already_exists,dry_run,dry_run,dry_run,dry_run
5,CL_MLP_PAO_IF_Ortho_26_01_23_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,already_exists,dry_run,dry_run,dry_run,dry_run
6,CL_MLP_PAO_IF_Ortho_26_02_04_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,already_exists,dry_run,dry_run,dry_run,dry_run
7,CL_MLP_PAO_IF_Ortho_26_02_20_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,already_exists,dry_run,dry_run,dry_run,dry_run
8,CL_MLP_PAO_IF_Ortho_26_02_26_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,already_exists,dry_run,dry_run,dry_run,dry_run
9,CL_MLP_PAO_IF_Ortho_26_03_07_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,already_exists,dry_run,dry_run,dry_run,dry_run


## 2. Preparar query y evitar duplicados en el indice

In [6]:
def quote_sql_text(value):
    return str(value).replace("'", "''")


def sql_in_clause(field_name, values):
    values = [value for value in values if value]
    if not values:
        return '1 = 0'
    joined = ', '.join(f"'{quote_sql_text(value)}'" for value in values)
    return f"{field_name} IN ({joined})"


def existing_names(feature_class, names):
    if not names:
        return set()
    where = sql_in_clause('Name', names)
    return {row[0] for row in arcpy.da.SearchCursor(feature_class, ['Name'], where_clause=where) if row[0]}


existing_target_names = existing_names(FC_FOOTPRINT_INDICE, names_from_stage_2) if SKIP_EXISTING_TARGET_NAMES else set()
names_to_append = [name for name in names_from_stage_2 if name not in existing_target_names]
where_mosaic_names = sql_in_clause('Name', names_to_append)

print(f'Ya existentes en indice: {len(existing_target_names)}')
print(f'Nombres a exportar y anexar: {len(names_to_append)}')
print('Where mosaic:', where_mosaic_names[:500] + ('...' if len(where_mosaic_names) > 500 else ''))

Ya existentes en indice: 7
Nombres a exportar y anexar: 26
Where mosaic: Name IN ('CL_MLP_PAO_IF_Ortho_25_12_10_MonteAranda-NSTC-Km-84p2-a-82p3', 'CL_MLP_PAO_IF_Ortho_25_12_20_MonteAranda-NSTC-Km-84p2-a-82p3', 'CL_MLP_PAO_IF_Ortho_25_12_28_MonteAranda-NSTC-Km-84p2-a-82p3', 'CL_MLP_PAO_IF_Ortho_26_01_03_MonteAranda-NSTC-Km-84p2-a-82p3', 'CL_MLP_PAO_IF_Ortho_26_01_23_MonteAranda-NSTC-Km-84p2-a-82p3', 'CL_MLP_PAO_IF_Ortho_26_02_04_MonteAranda-NSTC-Km-84p2-a-82p3', 'CL_MLP_PAO_IF_Ortho_26_02_20_MonteAranda-NSTC-Km-84p2-a-82p3', 'CL_MLP_PAO_IF_Ortho_26_02_26_MonteAranda-N...


## 3. Exportar footprints desde mosaic dataset y anexar al indice

In [7]:
field_renames = {
    'FechaAdqui': 'Fecha_Adqu',
    'FechaCarga': 'Fecha_Publ',
    'NombreVuelo': 'Nombre_de_Vuelo',
    'ProductName': 'ProductNam',
}


def ensure_file_gdb(gdb_path):
    gdb_path = Path(gdb_path)
    gdb_path.parent.mkdir(parents=True, exist_ok=True)
    if not arcpy.Exists(str(gdb_path)):
        arcpy.management.CreateFileGDB(str(gdb_path.parent), gdb_path.stem)
    return gdb_path


def rename_fields_if_present(feature_class, rename_map):
    existing = {field.name.lower(): field.name for field in arcpy.ListFields(feature_class)}
    applied = []
    skipped = []
    for old_name, new_name in rename_map.items():
        resolved_old = existing.get(old_name.lower())
        if not resolved_old:
            skipped.append((old_name, new_name, 'campo_origen_no_existe'))
            continue
        if new_name.lower() in existing and resolved_old.lower() != new_name.lower():
            skipped.append((old_name, new_name, 'campo_destino_ya_existe'))
            continue
        print(f'{resolved_old} ==> {new_name}')
        arcpy.management.AlterField(feature_class, resolved_old, new_name, new_name)
        applied.append((resolved_old, new_name, 'renombrado'))
        existing = {field.name.lower(): field.name for field in arcpy.ListFields(feature_class)}
    return applied, skipped


append_count = 0
export_count = 0
fc_footprint_temp = str(Path(TEMP_GDB) / FC_FOOTPRINT_TEMP_NAME)

if not names_to_append:
    print('No hay nombres nuevos para anexar al indice.')
elif DRY_RUN:
    print('[DRY_RUN] Se exportarian y anexarian footprints para:', len(names_to_append))
else:
    ensure_file_gdb(TEMP_GDB)
    if arcpy.Exists(fc_footprint_temp):
        arcpy.management.Delete(fc_footprint_temp)

    arcpy.ExportMosaicDatasetGeometry_management(
        PATH_MOSAIC_DATASET,
        out_feature_class=fc_footprint_temp,
        where_clause=where_mosaic_names,
    )
    export_count = int(arcpy.management.GetCount(fc_footprint_temp)[0])
    print(f'Footprints exportados: {export_count}')

    applied_renames, skipped_renames = rename_fields_if_present(fc_footprint_temp, field_renames)
    print('Renombres aplicados:', applied_renames)
    print('Renombres omitidos:', skipped_renames)

    arcpy.management.Append(fc_footprint_temp, FC_FOOTPRINT_INDICE, 'NO_TEST', '')
    append_count = export_count
    print(f'Footprints anexados al indice: {append_count}')

Footprints exportados: 24
FechaAdqui ==> Fecha_Adqu
FechaCarga ==> Fecha_Publ
NombreVuelo ==> Nombre_de_Vuelo
ProductName ==> ProductNam
Renombres aplicados: [('FechaAdqui', 'Fecha_Adqu', 'renombrado'), ('FechaCarga', 'Fecha_Publ', 'renombrado'), ('NombreVuelo', 'Nombre_de_Vuelo', 'renombrado'), ('ProductName', 'ProductNam', 'renombrado')]
Renombres omitidos: []
Footprints anexados al indice: 24


## 4. Recalcular Nombre_de_Vuelo en el indice

La regla sigue el notebook 009: ordenar por `Name ASC`, contar registros con `Sector` y `Fecha_Adqu`, y asignar `NNN_yy_mm_dd_sector`.

In [8]:
def recalculate_nombre_vuelo(feature_class, dry_run=True):
    cols = ['Nombre_de_Vuelo', 'Sector', 'Fecha_Adqu']
    updates = []
    with arcpy.da.UpdateCursor(
        feature_class,
        cols,
        where_clause=None,
        sql_clause=(None, 'ORDER BY Name ASC'),
    ) as cursor:
        cnt = 0
        for row in cursor:
            if row[1] and row[2]:
                cnt += 1
                fecha = pd.to_datetime(row[2]).strftime('%y_%m_%d')
                numero = str(cnt).zfill(3)
                sector = row[1]
                new_name = f'{numero}_{fecha}_{sector}'
                updates.append(new_name)
                if not dry_run:
                    row[0] = new_name
                    cursor.updateRow(row)
    return updates


if RECALCULATE_ALL_NOMBRE_VUELO:
    nombre_vuelo_updates = recalculate_nombre_vuelo(FC_FOOTPRINT_INDICE, dry_run=DRY_RUN)
else:
    nombre_vuelo_updates = []

print(f'Nombres de vuelo calculados: {len(nombre_vuelo_updates)}')
print('Muestra:')
for value in nombre_vuelo_updates[:30]:
    print(value)

Nombres de vuelo calculados: 505
Muestra:
001_25_01_08_EB1
002_25_02_26_RutaD835
003_25_03_10_NSTC_116a118
004_25_03_10_NSTC_118a120
005_25_03_12_SRA2
006_25_03_17_NSTC_Km_120_a_121
007_25_03_19_Orejas16_Ruta-D-865
008_25_03_19_Orejas17_Ruta-D-865
009_25_03_19_Orejas18
010_25_03_31_EB2
011_25_03_31_EV2
012_25_04_02_DME-13
013_25_04_02_Patio-Acopio-17
014_25_04_02_Ruta-SE-a-DME-13
015_25_04_03_Area-patio-19b-y-armado
016_25_04_03_Subestacion-El-Mauro
017_25_04_09_Cachimba_de_Bajo_Camisas_ED2
018_25_04_10_Sector_Pupio_I_Area_1
019_25_05_07_Monte-Aranda-84p2-a-82p3
020_25_05_07_Monte_Aranda_82p3_a_80p7
021_25_05_12_EDT
022_25_05_12_IIFF15-Campamento-Tipay
023_25_05_15_DME9_PA12_IIFF8
024_25_05_15_ED1_IIFF7_DME8
025_25_05_15_EM2_PA11_01
026_25_05_29_DME5A_DME17_a_NSTC_78p9
027_25_06_02_ByPassDrenes_Estacion_Drenes_El_Mauro
028_25_06_02_Estacion_Drenes_El_Mauro
029_25_06_04_CaminoAcceso03_LasAnimas
030_25_06_04_EM3


## 5. Actualizar NombreVuelo en el mosaic dataset

In [9]:
def target_nombre_vuelo_lookup(feature_class, names):
    if not names:
        return {}
    where = sql_in_clause('Name', names)
    result = {}
    with arcpy.da.SearchCursor(feature_class, ['Name', 'Nombre_de_Vuelo'], where_clause=where) as cursor:
        for name, nombre_vuelo in cursor:
            if name and nombre_vuelo:
                result[name] = nombre_vuelo
    return result


mosaic_nombre_vuelo_updates = []
if UPDATE_MOSAIC_NOMBRE_VUELO:
    lookup = target_nombre_vuelo_lookup(FC_FOOTPRINT_INDICE, names_from_stage_2)
    where = sql_in_clause('Name', list(lookup.keys()))
    print(f'Registros con Nombre_de_Vuelo para actualizar mosaico: {len(lookup)}')
    if DRY_RUN:
        display(pd.DataFrame([{'Name': key, 'NombreVuelo': value} for key, value in lookup.items()]).head(30))
    elif lookup:
        with arcpy.da.UpdateCursor(PATH_MOSAIC_DATASET, ['Name', 'NombreVuelo'], where_clause=where) as cursor:
            for row in cursor:
                new_value = lookup.get(row[0])
                if new_value:
                    old_value = row[1]
                    row[1] = new_value
                    cursor.updateRow(row)
                    mosaic_nombre_vuelo_updates.append({'Name': row[0], 'old': old_value, 'new': new_value})

print(f'NombreVuelo actualizados en mosaico: {len(mosaic_nombre_vuelo_updates)}')

Registros con Nombre_de_Vuelo para actualizar mosaico: 31
NombreVuelo actualizados en mosaico: 31


## 6. Exportar resumen

In [10]:
run_timestamp = pd.to_datetime('now').strftime('%Y%m%d_%H%M%S')
summary_df = pd.DataFrame([
    {'metric': 'run_timestamp', 'value': run_timestamp},
    {'metric': 'dry_run', 'value': DRY_RUN},
    {'metric': 'load_results_csv', 'value': str(LOAD_RESULTS_CSV)},
    {'metric': 'mosaic_dataset', 'value': PATH_MOSAIC_DATASET},
    {'metric': 'fc_footprint_indice', 'value': FC_FOOTPRINT_INDICE},
    {'metric': 'temp_gdb', 'value': str(TEMP_GDB)},
    {'metric': 'stage_2_names_count', 'value': len(names_from_stage_2)},
    {'metric': 'existing_target_names_count', 'value': len(existing_target_names)},
    {'metric': 'names_to_append_count', 'value': len(names_to_append)},
    {'metric': 'export_count', 'value': export_count},
    {'metric': 'append_count', 'value': append_count},
    {'metric': 'nombre_vuelo_calculated_count', 'value': len(nombre_vuelo_updates)},
    {'metric': 'mosaic_nombre_vuelo_updated_count', 'value': len(mosaic_nombre_vuelo_updates)},
])

summary_csv = OUTPUT_DIR / '00_summary.csv'
stage_2_names_csv = OUTPUT_DIR / '01_stage_2_names.csv'
names_to_append_csv = OUTPUT_DIR / '02_names_to_append.csv'
mosaic_nombre_vuelo_csv = OUTPUT_DIR / '03_mosaic_nombre_vuelo_updates.csv'

summary_df.to_csv(summary_csv, index=False, encoding='utf-8-sig')
pd.DataFrame({'Name': names_from_stage_2}).to_csv(stage_2_names_csv, index=False, encoding='utf-8-sig')
pd.DataFrame({'Name': names_to_append}).to_csv(names_to_append_csv, index=False, encoding='utf-8-sig')
pd.DataFrame(mosaic_nombre_vuelo_updates).to_csv(mosaic_nombre_vuelo_csv, index=False, encoding='utf-8-sig')

display(summary_df)
print('Outputs exportados en:', OUTPUT_DIR)

,metric,value
0,run_timestamp,20260619_090620
1,dry_run,False
2,load_results_csv,c:\Users\esrlrivero_adm\Documents\Geosupport\a...
3,mosaic_dataset,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proye...
4,fc_footprint_indice,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\...
5,temp_gdb,c:\Users\esrlrivero_adm\Documents\Geosupport\a...
6,stage_2_names_count,33
7,existing_target_names_count,7
8,names_to_append_count,26
9,export_count,24


Outputs exportados en: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\flujo_geosupport_etapas\outputs\etapa_03_actualizar_footprints_indice
